# S50_03 — LLM Monitoring and Observability

LLM applications fail in ways classical ML systems don't: hallucinations, prompt injections, context stuffing, unexpected refusals. Monitoring requires observability beyond accuracy metrics.

## What to monitor

| Category | Metrics |
|----------|--------|
| **Latency** | TTFT (time to first token), total latency, p50/p95/p99 |
| **Cost** | Input tokens, output tokens, cost per request |
| **Quality** | LLM-as-judge scores, user thumbs up/down, task completion rate |
| **Safety** | Harmful output rate, refusal rate, PII leakage |
| **Reliability** | Error rate, timeout rate, retry rate |

In [ ]:
import anthropic
import time
import json
from dataclasses import dataclass, field, asdict
from typing import Optional

@dataclass
class LLMTrace:
    request_id: str
    model: str
    prompt: str
    response: str
    input_tokens: int
    output_tokens: int
    latency_s: float
    cost_usd: float
    error: Optional[str] = None
    metadata: dict = field(default_factory=dict)

class LLMLogger:
    """Simple wrapper that logs every LLM call."""
    
    # claude-haiku-4-5 pricing (per million tokens)
    PRICING = {
        'claude-haiku-4-5-20251001': {'input': 1.00, 'output': 5.00},
        'claude-sonnet-4-6': {'input': 3.00, 'output': 15.00},
    }
    
    def __init__(self):
        self.client = anthropic.Anthropic()
        self.traces = []
    
    def call(self, model, messages, system=None, max_tokens=512, metadata=None):
        import uuid
        request_id = str(uuid.uuid4())[:8]
        start = time.time()
        
        try:
            kwargs = dict(model=model, max_tokens=max_tokens, messages=messages)
            if system:
                kwargs['system'] = system
            
            response = self.client.messages.create(**kwargs)
            latency = time.time() - start
            
            pricing = self.PRICING.get(model, {'input': 0, 'output': 0})
            cost = (
                response.usage.input_tokens * pricing['input'] / 1e6 +
                response.usage.output_tokens * pricing['output'] / 1e6
            )
            
            trace = LLMTrace(
                request_id=request_id,
                model=model,
                prompt=messages[-1]['content'][:200],
                response=response.content[0].text,
                input_tokens=response.usage.input_tokens,
                output_tokens=response.usage.output_tokens,
                latency_s=round(latency, 3),
                cost_usd=round(cost, 6),
                metadata=metadata or {},
            )
            self.traces.append(trace)
            return response.content[0].text
        
        except Exception as e:
            trace = LLMTrace(
                request_id=request_id, model=model,
                prompt=messages[-1]['content'][:200],
                response='', input_tokens=0, output_tokens=0,
                latency_s=round(time.time() - start, 3),
                cost_usd=0, error=str(e),
            )
            self.traces.append(trace)
            raise
    
    def summary(self):
        if not self.traces:
            return 'No traces'
        total_cost = sum(t.cost_usd for t in self.traces)
        avg_latency = sum(t.latency_s for t in self.traces) / len(self.traces)
        total_tokens = sum(t.input_tokens + t.output_tokens for t in self.traces)
        return {
            'n_calls': len(self.traces),
            'total_tokens': total_tokens,
            'total_cost_usd': round(total_cost, 6),
            'avg_latency_s': round(avg_latency, 3),
            'errors': sum(1 for t in self.traces if t.error),
        }

logger = LLMLogger()
r1 = logger.call('claude-haiku-4-5-20251001', [{'role': 'user', 'content': 'What is overfitting?'}], metadata={'use_case': 'demo'})
r2 = logger.call('claude-haiku-4-5-20251001', [{'role': 'user', 'content': 'What is gradient descent?'}], metadata={'use_case': 'demo'})

print(json.dumps(logger.summary(), indent=2))

## LangSmith — production tracing

In [ ]:
# LangSmith traces every LangChain/LangGraph call automatically
# pip install langsmith

langsmith_setup = '''
import os
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = "ls-..."
os.environ["LANGCHAIN_PROJECT"] = "my-project"

# That's it — all LangChain calls are now traced automatically
from langchain_anthropic import ChatAnthropic
llm = ChatAnthropic(model="claude-haiku-4-5-20251001")

# This call appears in LangSmith dashboard with:
# - Full input/output
# - Token counts and latency
# - Error traces if it fails
# - Links between parent/child chains
result = llm.invoke("What is attention?")
'''

print('LangSmith tracing setup:')
print(langsmith_setup)
print('Dashboard: https://smith.langchain.com')

## Output quality monitoring

In [ ]:
# Automated quality checks using LLM-as-judge
def check_output_quality(prompt, response, checks=None):
    """Run a set of automated quality checks on an LLM response."""
    if checks is None:
        checks = ['relevance', 'factual_grounding', 'safe']
    
    client = anthropic.Anthropic()
    check_prompt = f"""
Evaluate this AI response. Respond ONLY with valid JSON.

User prompt: {prompt}
AI response: {response}

Checks to perform: {checks}

For each check, provide: {{"passed": bool, "score": 1-5, "issue": str or null}}
Return: {{{', '.join(f'"{c}": {{...}}' for c in checks)}}}
"""
    msg = client.messages.create(
        model='claude-haiku-4-5-20251001',
        max_tokens=512,
        messages=[{'role': 'user', 'content': check_prompt}],
    )
    return json.loads(msg.content[0].text)

# Monitor a sample of production calls
sample_prompt = 'What is the capital of Australia?'
sample_response = 'The capital of Australia is Canberra.'

quality = check_output_quality(sample_prompt, sample_response)
print(json.dumps(quality, indent=2))

## Monitoring stack options

| Tool | Type | Best for |
|------|------|----------|
| **LangSmith** | Managed SaaS | LangChain apps, full trace viz |
| **Langfuse** | Open-source / cloud | Self-hosted, GDPR-friendly |
| **Weights & Biases** | MLOps + LLM | Combined training + inference monitoring |
| **Arize AI** | Enterprise | Drift detection, bias monitoring |
| **OpenTelemetry** | Open standard | Custom pipelines, any stack |
| **Prometheus + Grafana** | Infrastructure | Latency, error rate, throughput |

Next: [S50_04_cost_optimization.ipynb](./S50_04_cost_optimization.ipynb)